# LLM From Scratch

In [3]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
import numpy as np
import math

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [5]:
with open("sherlock_holmes.txt", "r", encoding="utf-8") as f:
    text = f.read()
    print(len(text))

562212


In [47]:
chars = sorted(set(text))
vocab_size = len(chars)
vocab_size

88

In [7]:
# character level encoding

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [8]:
encode('jack reacher')

[58, 49, 51, 59, 1, 66, 53, 49, 51, 56, 53, 66]

In [9]:
decode([58, 49, 51, 59, 1, 66, 53, 49, 51, 56, 53, 66])

'jack reacher'

In [10]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([41, 56, 53,  1, 22, 52, 70, 53, 62, 68, 69, 66, 53, 67,  1, 63, 54,  1,
        40, 56, 53, 66, 60, 63, 51, 59,  1, 29, 63, 60, 61, 53, 67,  0,  0, 50,
        73,  1, 22, 66, 68, 56, 69, 66,  1, 24, 63, 62, 49, 62,  1, 25, 63, 73,
        60, 53,  0,  0,  0, 24, 63, 62, 68, 53, 62, 68, 67,  0,  0,  1,  1,  1,
        30,  8,  1,  1,  1,  1,  1, 22,  1, 40, 51, 49, 62, 52, 49, 60,  1, 57,
        62,  1, 23, 63, 56, 53, 61, 57, 49,  0])


In [11]:
split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

### Bigram Language Model into a Neural Network

In [12]:
# this sequentially slides the window over the text 
# this can only be done by the CPU because it's sequential

block_size = 8
batch_size = 4

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target  = y[t]
    print(f"when input is {context}, target is {target}")

when input is tensor([41]), target is 56
when input is tensor([41, 56]), target is 53
when input is tensor([41, 56, 53]), target is 1
when input is tensor([41, 56, 53,  1]), target is 22
when input is tensor([41, 56, 53,  1, 22]), target is 52
when input is tensor([41, 56, 53,  1, 22, 52]), target is 70
when input is tensor([41, 56, 53,  1, 22, 52, 70]), target is 53
when input is tensor([41, 56, 53,  1, 22, 52, 70, 53]), target is 62


In [13]:
out = torch.zeros(6,6).masked_fill(torch.tril(torch.ones(6,6)) == 0, float('-inf'))
out

tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])

In [14]:
# we split this into multiple blocks and pass it to the GPU to compute it parallely
torch.exp(out)


tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [15]:
ten = torch.zeros(2, 3, 4)
ten

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [16]:
out = ten.transpose(0,2)
out

tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])

In [17]:
t1 = torch.tensor([1,2,3])
t2 = torch.tensor([4,6,8])
t3 = torch.tensor([8,1,5])

tr = torch.stack([t1, t2, t3])
tr

tensor([[1, 2, 3],
        [4, 6, 8],
        [8, 1, 5]])

In [18]:
sample = torch.tensor([19.0, 29.0, 39.0])
linear = torch.nn.Linear(3, 3, bias=False)
print(linear(sample))

tensor([ 0.2712,  7.3966, -9.4153], grad_fn=<SqueezeBackward4>)


In [19]:
emb = nn.Embedding(26, 100)
input_indices = torch.LongTensor([1,5,3,2])
emb_output = emb(input_indices)
emb_output.shape

torch.Size([4, 100])

In [20]:
t1 = torch.Tensor([[1,2,3],[3,4,5]])
t2 = torch.Tensor([[73,62],[34,59],[33,95]])

t1 @ t2

tensor([[240., 465.],
        [520., 897.]])

In [21]:
block_size = 8
batch_size = 4

k = int(0.8 * len(data))
train_data = data[:k]
val_data   = data[k:]

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    x, y = x.to(device), y.to(device)

    return x, y

x, y = get_batch('train')

In [22]:
print(x)
print(y)

tensor([[53,  1, 49, 62, 52,  1, 71, 49],
        [73,  1, 67, 53, 66, 70, 53, 52],
        [66, 63, 69, 55, 56, 68,  1, 57],
        [68, 57, 60, 60,  1, 61, 63, 66]], device='cuda:0')
tensor([[ 1, 49, 62, 52,  1, 71, 49, 67],
        [ 1, 67, 53, 66, 70, 53, 52,  0],
        [63, 69, 55, 56, 68,  1, 57, 68],
        [57, 60, 60,  1, 61, 63, 66, 53]], device='cuda:0')


In [61]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # table shape : vocab_size x vocab_size
    
    def forward(self, index, targets=None):
        
        logits = self.token_embedding_table(index)

        if targets is None: 
            loss = None 
        else: 
            # Batch, Time, Channels (vocab_size)
            """
            because we need to pay attention to the Channels
            we can blend Batch and Time together and 
            as long as logits and targets have the same 
            batch and time, we should be alright
            """
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss 


    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index) 
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # get probabilities
            index_next = torch.multinomial(probs, num_samples=1) # sample from distribution
            index = torch.cat((index, index_next), dim=-1) # (B, T+1)
        return index 

In [65]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


9m3YCgKécTk£dœ’8Cé,p(JI??l7Llèœ&æ;VyJ874&(x&_,5R0tld½-nWœyR2(KæPm3-m3V1PG;SisH‘S7eUuC,T;QZ3m3CjfDàL0Gæ(2w7tXè2màuRcI3Oh—PSx psRs!8aRka
x½H03sL07Ld‘h7L9sx1FP”nn”1;fXF,fWF“NGœGyèqBæ8Cg£xg-(VFH”lzâqBq2dâo0“ed34wxFP?PNU,xz?x
7h—eHàq”B3h”NI8?ZfXfvà7Ls?sLJYqUd)CCjK‘(gd GT5aœmi(eedè,Y‘2cO‘Aæ,)0HàCq6.:æ,£(“clIqn6_xG lèw&NlIubv_’!?AKKpKn,(V-ètZ3YitrJ&v½7tv.uU
X”tlicIr5f7RæD0HIR) QBvyk“FA’!dD,FsrVHEG!9OJXdJ8NG-Ta?bqàQRWe’g5’â.:½CAaF!9m_Q£œ xtr½½DQJ)?-m(H9pæ3½tW9:‘yào?vjâP“èy9;PLORo0GyMkèBæ;9m3J½XIœO2y“r’’


In [36]:
T = torch.Tensor([2,3,6,4])
T.view(2,2)

tensor([[2., 3.],
        [6., 4.]])